In [1]:
import pandas as pd
import numpy as np

In [2]:
def hojas_a_df(ruta_archivo):
    """Lee el Excel y crea variables globales con prefijo df_"""
    from IPython import get_ipython
    dfs = pd.read_excel(ruta_archivo, sheet_name=None)
    shell = get_ipython()
    for nombre, df in dfs.items():
        nombre_var = 'df_' + nombre.replace(' ', '_').replace('-', '_')
        shell.user_ns[nombre_var] = df
    return dfs

hojas_a_df('data/materiales.xlsx')

{'Table of Contents':     Unnamed: 0         Table of Contents               Unnamed: 2
 0          NaN              40K Editions                      NaN
 1          NaN            Codex Releases                      NaN
 2          NaN               Craftworlds                      NaN
 3          NaN                  Drukhari                      NaN
 4          NaN                Harlequins                      NaN
 5          NaN                    Ynnari                      NaN
 6          NaN          Genestealer Cult                      NaN
 7          NaN                  Tyranids                      NaN
 8          NaN                   Necrons                      NaN
 9          NaN                      Orks                      NaN
 10         NaN                Tau Empire                      NaN
 11         NaN             Space Marines                      NaN
 12         NaN        Supplement Marines                      NaN
 13         NaN              Blood Angels

In [3]:
df_materiales = pd.DataFrame(columns=('mini', 'faction', 'material', 'edition', 'year'))
df_materiales

,mini,faction,material,edition,year


In [4]:
edition = df_40K_Editions.drop(columns=df_40K_Editions.columns[0])
edition = edition['Release'].tolist()
edition

['Warhammer 40K: Rogue Trader',
 'Warhammer 40K: 2nd Edition',
 'Warhammer 40K: 3rd Edition',
 'Warhammer 40K: 4th Edition',
 'Warhammer 40K: 5th Edition',
 'Warhammer 40K: 6th Edition',
 'Warhammer 40K: 7th Edition',
 'Warhammer 40K: 8th Edition',
 'Warhammer 40K: 9th Edition',
 'Warhammer 40K: 10th Edition']

In [5]:
faction = df_Table_of_Contents.drop(columns=df_Table_of_Contents.columns[[0, 2]])[2:30]
faction = faction['Table of Contents'].str.replace(' ', '_').tolist()
faction

['Craftworlds',
 'Drukhari',
 'Harlequins',
 'Ynnari',
 'Genestealer_Cult',
 'Tyranids',
 'Necrons',
 'Orks',
 'Tau_Empire',
 'Space_Marines',
 'Supplement_Marines',
 'Blood_Angels',
 'Dark_Angels',
 'Space_Wolves',
 'Deathwatch',
 'Grey_Knights',
 'Astra_Militarum',
 'Adepta_Sororitas',
 'Adeptus_Custodes',
 'Adeptus_Mechanicus',
 'Imperial_Assassins',
 'Imperial_Knights',
 'Inquisition',
 'Chaos_Daemons',
 'Chaos_Knights',
 'Chaos_Space_Marines',
 'Death_Guard',
 'Thousand_Sons']

In [6]:
def dateFormater(df, dateColumn):
    df[dateColumn] = pd.to_datetime(df[dateColumn])
    df['year'] = df[dateColumn].dt.year.astype('Int64')
    df.drop(dateColumn, axis=1, inplace=True)
    if 'day' in df.columns:
        df.drop('day', axis=1, inplace=True)
    if 'month' in df.columns:
        df.drop('month', axis=1, inplace=True)

In [7]:
def process_faction(df, faction_name):
    if 'Year-Month' in df.columns:
        dateFormater(df, 'Year-Month')
    
    df = df.rename(columns={'Release': 'mini', 'Current Material': 'material'})
    df = df.drop(columns=[c for c in ['Unnamed: 4'] if c in df.columns])

    df = df.dropna(subset=['mini'])

    mask = df['mini'].isin(edition)
    df['edition'] = df['mini'].where(mask).ffill()
    df = df[~df['mini'].str.contains('Warhammer')]
    df = df.replace(False, np.nan).dropna(how='all')

    df = df.reset_index(drop=True)
    df['material'] = df['material'].fillna('Plastic')

    df['faction'] = faction_name.replace('df_', '')
    df = df[df_materiales.columns]

    return df

In [8]:
def creator(factionList, all_dfs):
    df_list = []
    
    for f in factionList:
        df = all_dfs[f'df_{f}']
        processed_df = process_faction(df, f)
        df_list.append(processed_df)
    
    return pd.concat(df_list, ignore_index=True)

df_combined = creator(faction, globals())

In [ ]:
df_combined.to_csv('clear_data/df_materials.csv', index=False)
df_combined

,mini,faction,material,edition,year
0,Baharroth Wings,Craftworlds,Metal,Warhammer 40K: Rogue Trader,1990
1,Warp Spiders,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994
2,Avatar of Khaine,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994
3,Asurmen,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994
4,Karandras,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994
...,...,...,...,...,...
977,Ahriman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2016
978,Tzaangor Enlightened,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017
979,Tzaangor Skyfires,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017
980,Tzaangor Shaman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017
